# 05 - Anomalies, schedule irregularity and day archetypes (M3)

Review aid for the M3 scripts. Run them in this order, then open this notebook:

```
python src/features.py          # adds timing + activity features
python src/m3_anomalies.py      # out-of-fold residuals, flags (needed by archetypes)
python src/m3_irregularity.py   # pre-registered; refuses to run unless preregistration_m3.json is committed
python src/m3_archetypes.py
python src/m3_figures.py
```

Written summary: `results/m3_summary.txt`. Cells marked **private** read git-ignored per-cycle files; don't paste their output anywhere public.

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.precision", 2)
pd.set_option("display.width", 200)
R = "../results/"
print(open(R + "m3_summary.txt").read()[:1200])

## 1. Anomaly detector

Residual = actual − the formula model's out-of-fold prediction. Flag when |residual| > 2.5 × robust SD (1.4826·MAD).
The question to ask of this section: are the anomalies random, or does the model miss a *shape*?

In [ ]:
pd.read_csv(R + "m3_anomaly_summary.csv", index_col=0)

In [ ]:
display(Image(R + "figures/m3_residual_by_input.png"))
pd.read_csv(R + "m3_anomaly_residual_by_input.csv")

In [ ]:
# which group dominated the prediction, flagged vs all cycles
pd.read_csv(R + "m3_anomaly_groups.csv").sort_values("top_group_freq_flagged", ascending=False)

In [ ]:
# PRIVATE: the flagged cycles, what the model expected and what it relied on
anom = pd.read_csv("../data/processed/m3_anomalies.csv")
anom[["cycle_start_local", "is_test", "actual", "predicted", "residual", "baseline_B",
      "top_groups", "most_unusual_input", "most_unusual_robust_z"]]

## 2. Schedule irregularity (pre-registered)

Hypothesis, controls, inference and pass rule are fixed in `preregistration_m3.json` (committed before the run).
**Total effect** = HRV/RHR not controlled (headline, pass/fail). **Direct effect** = HRV/RHR controlled (reported alongside).

In [ ]:
print(open("../preregistration_m3.json").read())

In [ ]:
display(Image(R + "figures/m3_irregularity.png"))
pd.read_csv(R + "m3_irregularity_results.csv")

In [ ]:
# the timing features themselves, and how they relate to sleep and HRV
pd.read_csv(R + "m3_irregularity_timing_describe.csv", index_col=0)

## 3. Day archetypes

k-means on z-scored [strain, number of activities, hours slept (the sleep that opened the day)], k by silhouette.
Compared on the **next** cycle: raw recovery, and the model's residual (what the next cycle's inputs don't explain).

In [ ]:
pd.read_csv(R + "m3_archetypes_meta.csv", index_col=0)

In [ ]:
display(Image(R + "figures/m3_archetypes.png"))
pd.read_csv(R + "m3_archetypes.csv")

In [ ]:
# PRIVATE: which archetype each day fell in
days = pd.read_csv("../data/processed/m3_archetype_days.csv")
days.groupby("archetype")[["strain", "n_activities", "total_sleep_hours", "activity_minutes",
                           "next_recovery", "next_residual", "next_sleep_hours"]].median()